# ADMM Direct Day Optimization

This notebook builds a direct day-ahead optimization window, solves the same convex model with a centralized LP benchmark and an ADMM decomposition, and compares the resulting economics and transformer export safety diagnostics.


In [ ]:
from dataclasses import replace
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "configs").exists():
    repo_root = repo_root.parent
if not (repo_root / "configs").exists():
    raise RuntimeError("Could not locate the project root from the notebook working directory.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import configs as configs_pkg
from configs import compose_experiment_config
from scripts.builder import build_env
from scripts.utils import grid_notebook_workflow as grid_nb
from scripts.utils import admm_direct_notebook_helpers as direct_nb
from scripts.utils.project_paths import project_root as resolve_project_root

configs_pkg = importlib.reload(configs_pkg)
grid_nb = importlib.reload(grid_nb)
direct_nb = importlib.reload(direct_nb)


In [ ]:
PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data"

TEST_START_DATE = "2020-06-01"
TEST_END_DATE = "2020-06-05"
PREDICTION_MODE = "perfect"

RHO_ADAPTATION = "residual_balancing"
MAX_ITERS = 300
PRIMAL_TOL = 1e-3
DUAL_TOL = 1e-3


In [ ]:
cfg = compose_experiment_config(
    profile="base",
    algorithm="MATD3",
    model_family="mlp",
    data_dir=DATA_DIR,
)
grid_nb.apply_notebook_experiment_settings(
    cfg,
    prediction_mode=PREDICTION_MODE,
    test_start_date=TEST_START_DATE,
    test_end_date=TEST_END_DATE,
)
cfg.env.episode_limit = int(round(24.0 / cfg.env.dt))
assert abs(cfg.env.dt * cfg.env.episode_limit - 24.0) <= 1e-9
env = build_env(cfg, mode="test")

display(
    pd.Series(
        {
            "test_start_date": TEST_START_DATE,
            "test_end_date": TEST_END_DATE,
            "prediction_mode": PREDICTION_MODE,
            "dt_hours": cfg.env.dt,
            "episode_limit": cfg.env.episode_limit,
            "n_agents": cfg.env.num_agents,
            "n_days": int((pd.Timestamp(TEST_END_DATE) - pd.Timestamp(TEST_START_DATE)).days + 1),
        },
        name="admm_direct_day_config",
    )
)


In [ ]:
problem_data = direct_nb.build_direct_day_problem_data(
    cfg,
    test_start_date=TEST_START_DATE,
    test_end_date=TEST_END_DATE,
)
surrogate = direct_nb.build_direct_day_trafo_surrogate(cfg, env, problem_data)
baseline_solution = direct_nb.compute_direct_day_baseline(problem_data, surrogate)

if baseline_solution.max_export_violation_kw > 0.1 * surrogate.trafo_limit_kw:
    print("Warning: baseline export overload is large relative to the transformer limit; surrogate extrapolation may be significant.")
if not np.any(np.asarray(surrogate.export_overload_mask, dtype=bool)):
    print("No active export overload appears in the baseline window; this run is mainly a sanity check.")

display(
    pd.Series(
        {
            "window_steps": problem_data.horizon,
            "n_days": int(problem_data.horizon / cfg.env.episode_limit),
            "wholesale_price_min": float(np.min(problem_data.wholesale_price_eur_per_kwh)),
            "import_price_min": float(np.min(problem_data.import_price_eur_per_kwh)),
            "export_subsidy": float(problem_data.export_subsidy_eur_per_kwh),
            "min_import_minus_subsidy": float(np.min(problem_data.import_price_eur_per_kwh - problem_data.export_subsidy_eur_per_kwh)),
            "trafo_limit_kw": float(surrogate.trafo_limit_kw),
            "baseline_max_export_violation_kw": float(baseline_solution.max_export_violation_kw),
        },
        name="admm_direct_day_window_summary",
    )
)


In [ ]:
rho_init = float(
    np.mean(problem_data.import_price_eur_per_kwh) * problem_data.dt_hours
    / max(1.0, float(np.mean(np.abs(surrogate.alpha_netload_window_kw * (problem_data.load_kw - problem_data.pv_kw)))))
)

centralized_solution = direct_nb.solve_direct_day_centralized(problem_data, surrogate)
centralized_step_df, centralized_grid_df = direct_nb.compute_direct_day_grid_profile(env, problem_data, centralized_solution)
centralized_solution = replace(
    centralized_solution,
    pp_root_p_kw=centralized_step_df["pp_root_p_kw"].to_numpy(dtype=np.float32, copy=True),
)

admm_result = direct_nb.solve_direct_day_admm(
    problem_data,
    surrogate,
    rho_init=rho_init,
    rho_adaptation=RHO_ADAPTATION,
    max_iters=MAX_ITERS,
    primal_tol=PRIMAL_TOL,
    dual_tol=DUAL_TOL,
    centralized_objective_eur=centralized_solution.objective_eur,
)
admm_step_df, admm_grid_df = direct_nb.compute_direct_day_grid_profile(env, problem_data, admm_result.solution)
admm_result = replace(
    admm_result,
    solution=replace(
        admm_result.solution,
        pp_root_p_kw=admm_step_df["pp_root_p_kw"].to_numpy(dtype=np.float32, copy=True),
    ),
)


In [ ]:
summary_df = pd.DataFrame(
    {
        "Baseline": direct_nb.summarize_direct_day_solution(baseline_solution, surrogate),
        "Centralized": direct_nb.summarize_direct_day_solution(centralized_solution, surrogate),
        "ADMM": direct_nb.summarize_direct_day_solution(admm_result.solution, surrogate),
    }
).T
display(summary_df)

display(
    pd.Series(
        {
            "converged": bool(admm_result.converged),
            "iterations": int(admm_result.iterations),
            "final_primal_residual": float(admm_result.final_primal_residual),
            "final_dual_residual": float(admm_result.final_dual_residual),
            "objective_gap_vs_centralized_eur": float(admm_result.objective_gap_vs_centralized_eur),
            "objective_gap_vs_centralized_pct": float(admm_result.objective_gap_vs_centralized_pct),
        },
        name="admm_direct_day_admm_summary",
    )
)


In [ ]:
# Main plots focused on the ADMM solution.
direct_nb.plot_direct_day_convergence(admm_result)
direct_nb.plot_direct_day_power_energy_balance(
    admm_result.solution,
    problem_data,
    baseline_solution=baseline_solution,
    label="ADMM",
)
direct_nb.plot_direct_day_voltage_profile(
    admm_result.solution,
    problem_data,
    env,
    cfg,
    label="ADMM",
    grid_df=admm_grid_df,
)
direct_nb.plot_direct_day_net_load(
    admm_result.solution,
    problem_data,
    surrogate,
    baseline_solution=baseline_solution,
    centralized_solution=centralized_solution,
    label="ADMM",
)

# Diagnostics.
direct_nb.plot_direct_day_root_p_comparison(
    centralized_solution,
    problem_data,
    surrogate,
    baseline_solution=baseline_solution,
    label="Centralized",
)
direct_nb.plot_direct_day_root_p_comparison(
    admm_result.solution,
    problem_data,
    surrogate,
    baseline_solution=baseline_solution,
    label="ADMM",
)
if False:
    direct_nb.plot_direct_day_soc_and_battery(admm_result.solution, problem_data)
plt.show()
